In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-08-05 23:00:31--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.3’

input.txt.3         100%[===================>]   1.06M   954KB/s    in 1.1s    

2026-08-05 23:00:33 (954 KB/s) - ‘input.txt.3’ saved [1115394/1115394]



In [2]:
import torch
print(f"is GPU available? {torch.cuda.is_available()}")

/home/yanfu-ou/Documents/SELF-TEACHING/learn-gpt-from-scratch/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


is GPU available? True


In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read() 
print("length of the dataset in characters: ", len(text))

length of the dataset in characters:  1115394


In [4]:
# here are all the unique characters in the shakesphere text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print("vocab_size: ", vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab_size:  65


# Tokenizer

In [5]:
# create a mapping from string to index(stoi) and from index to string(itos)
stoi = { char:i for i, char in enumerate(chars)}
itos = { i:char for i, char in enumerate(chars)}
encode = lambda word: [stoi[c] for c in word]  
decode = lambda nums: [itos[n] for n in nums]

print("Encoding: testing")
print("encoded: ", encode("testing")) # note that the first char and the  4th char are both 58 because they're the same char
print("decoded: ", decode(encode("testing"))) 

Encoding: testing
encoded:  [58, 43, 57, 58, 47, 52, 45]
decoded:  ['t', 'e', 's', 't', 'i', 'n', 'g']


In [6]:
# now the entire text is represented as a very large sequence of ints
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [7]:
# let's split our dataset up into train test split
n = int(0.9*len(data)) # first 90% will be train, rest will be val 
train_data = data[:n]
val_data = data[n:]

In [8]:
# we don't train it on the entire dataset at once, we train things block by block
block_size = 8 
train_data[:block_size+1] # first 9 characters in the sequence --> they all follow each other 

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [9]:
# goal: we want train the transformer to make prediction at each character/position, 
# given the previous characters as context, from 0 len up till block size  
# ex: to predict 56, the context is 18 + 47 
x = train_data[:block_size] 
y = train_data[1:block_size + 1] # target, aka what we're trying to predict
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"When the context is {context}, the target is {target}")


When the context is tensor([18]), the target is 47
When the context is tensor([18, 47]), the target is 56
When the context is tensor([18, 47, 56]), the target is 57
When the context is tensor([18, 47, 56, 57]), the target is 58
When the context is tensor([18, 47, 56, 57, 58]), the target is 1
When the context is tensor([18, 47, 56, 57, 58,  1]), the target is 15
When the context is tensor([18, 47, 56, 57, 58,  1, 15]), the target is 47
When the context is tensor([18, 47, 56, 57, 58,  1, 15, 47]), the target is 58


In [10]:
torch.manual_seed(1337)
batch_size = 4 # how many blocks to grab at once 
block_size = 8 # how many characters(or items) to grab in 1 batch 

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # syntax: torch.randint(high, size), where low = 0 by default
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    # print(x)
    # print(y)
    return x,y

xb, yb = get_batch("train")
print("inputs ", xb.shape)
print("targets ", yb.shape)

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        targets = yb[b, t]
        print(f"for context {context}, the targets are {targets}")

inputs  torch.Size([4, 8])
targets  torch.Size([4, 8])
for context tensor([24]), the targets are 43
for context tensor([24, 43]), the targets are 58
for context tensor([24, 43, 58]), the targets are 5
for context tensor([24, 43, 58,  5]), the targets are 57
for context tensor([24, 43, 58,  5, 57]), the targets are 1
for context tensor([24, 43, 58,  5, 57,  1]), the targets are 46
for context tensor([24, 43, 58,  5, 57,  1, 46]), the targets are 43
for context tensor([24, 43, 58,  5, 57,  1, 46, 43]), the targets are 39
for context tensor([44]), the targets are 53
for context tensor([44, 53]), the targets are 56
for context tensor([44, 53, 56]), the targets are 1
for context tensor([44, 53, 56,  1]), the targets are 58
for context tensor([44, 53, 56,  1, 58]), the targets are 46
for context tensor([44, 53, 56,  1, 58, 46]), the targets are 39
for context tensor([44, 53, 56,  1, 58, 46, 39]), the targets are 58
for context tensor([44, 53, 56,  1, 58, 46, 39, 58]), the targets are 1
for c

In [11]:
for row in range(xb.shape[0]):
    print(decode(xb[row].tolist()))

print("vocab size", vocab_size)

for row in range(yb.shape[0]):
    print(decode(yb[row].tolist()))

['L', 'e', 't', "'", 's', ' ', 'h', 'e']
['f', 'o', 'r', ' ', 't', 'h', 'a', 't']
['n', 't', ' ', 't', 'h', 'a', 't', ' ']
['M', 'E', 'O', ':', '\n', 'I', ' ', 'p']
vocab size 65
['e', 't', "'", 's', ' ', 'h', 'e', 'a']
['o', 'r', ' ', 't', 'h', 'a', 't', ' ']
['t', ' ', 't', 'h', 'a', 't', ' ', 'h']
['E', 'O', ':', '\n', 'I', ' ', 'p', 'a']


In [12]:
# learn cross entropy loss 
from torch.nn import functional as F
context = torch.tensor([[2.0, 1.0, 0.0], [0.2, 0.1, 0.0]])
goal = torch.tensor([2, 1])
print(f"context: {context.shape}, target: {goal.shape}")
loss = F.cross_entropy(context, goal) 
print(f"loss {loss}")

context: torch.Size([2, 3]), target: torch.Size([2])
loss 1.7547743320465088


In [13]:
print(torch.zeros((2, 2)))

tensor([[0., 0.],
        [0., 0.]])


In [14]:
print(xb)
print("-----")
print(yb)

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
-----
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [15]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class TestLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    def forward(self, idx):
        logits = self.token_embedding_table(idx)
        print("the logits are ", logits)
        return logits 

t = TestLanguageModel(4)
testing = torch.tensor([
    [1, 2, 0],
    [3, 0, 1]
])

print("token_embedding_table weights \n ", t.token_embedding_table.weight)
out = t(testing)
print("(B x T x C) ", out.shape)

token_embedding_table weights 
  Parameter containing:
tensor([[ 0.1808, -0.0700, -0.3596, -0.9152],
        [ 0.6258,  0.0255,  0.9545,  0.0643],
        [ 0.3612,  1.1679, -1.3499, -0.5102],
        [ 0.2360, -0.2398, -0.9211,  1.5433]], requires_grad=True)
the logits are  tensor([[[ 0.6258,  0.0255,  0.9545,  0.0643],
         [ 0.3612,  1.1679, -1.3499, -0.5102],
         [ 0.1808, -0.0700, -0.3596, -0.9152]],

        [[ 0.2360, -0.2398, -0.9211,  1.5433],
         [ 0.1808, -0.0700, -0.3596, -0.9152],
         [ 0.6258,  0.0255,  0.9545,  0.0643]]], grad_fn=<EmbeddingBackward0>)
(B x T x C)  torch.Size([2, 3, 4])


In [16]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # imagine a lookup table, where each row represents a character, 
        # and each column represents the prob that letter will be the next character 
        # aka - trainable lookup table 
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) # this should be 65x65

    def forward(self, idx, targets):
        # idx and targets are both (Batch size, Time/sequence) tensor of integers 
        # essentially performing a massive lookup for each token, and grabs the corresponding row 
        logits = self.token_embedding_table(idx) # (B, T, C)
        # if there's no target, then there's no loss and just gets the logits
        if targets is None:
            loss = None
        # if there's target, then calculate the loss 
        else:
            # print("logits shape: ", logits.shape)
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            # print("normalized logits shape", logits.shape)
            targets = targets.view(B * T)
            # print("normalized target shape", targets.shape)
            # loss is the cross entropy between logits and targets, and performs 2 steps 
                # 1 - softmax the logits so it becomes prob 
                # 2 - turn target integer into "The truth" using 1 hot encoding 
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for i in range(max_new_tokens):
            print(f"iter {i}, the idx: {idx}, decoded: {decode(idx[0].tolist())}")
            # gets predictions
            logits = self.token_embedding_table(idx)
            # focus only on the last time step
            logits = logits[:, -1, :]
            # apply softmax to get probabilities 
            probs = F.softmax(logits, dim=-1)
            # sample from distribution to pick the next character based on the normalized distribution from the logits 
            next_token = torch.multinomial(probs, num_samples=1)
            print(f"\t next token: {next_token[0]} {decode(next_token[0].tolist())} ")
            # append sampled index to the running sequence 
            idx = torch.cat((idx, next_token), dim=1)
        return idx

m = BigramLanguageModel(vocab_size)
# yb is just 1 shifted of xb, they're shape (B-batch size=4, T-time sequence=8)
out, loss = m(xb, yb)
print("out shape", out.shape)
print("cross entropy loss", loss)

# the idx used to jump start the generation. Aka the first token 
starter_idx = torch.zeros((1, 1), dtype=torch.long)
print("starter_idx", starter_idx)
generated_chars = m.generate(idx=starter_idx, max_new_tokens=10)
decoded_chars = decode(generated_chars[0].tolist())
print("decoded_chars", "".join(decoded_chars)) # jibberish because the model is untrained 

out shape torch.Size([32, 65])
cross entropy loss tensor(4.8786, grad_fn=<NllLossBackward0>)
starter_idx tensor([[0]])
iter 0, the idx: tensor([[0]]), decoded: ['\n']
	 next token: tensor([31]) ['S'] 
iter 1, the idx: tensor([[ 0, 31]]), decoded: ['\n', 'S']
	 next token: tensor([23]) ['K'] 
iter 2, the idx: tensor([[ 0, 31, 23]]), decoded: ['\n', 'S', 'K']
	 next token: tensor([21]) ['I'] 
iter 3, the idx: tensor([[ 0, 31, 23, 21]]), decoded: ['\n', 'S', 'K', 'I']
	 next token: tensor([41]) ['c'] 
iter 4, the idx: tensor([[ 0, 31, 23, 21, 41]]), decoded: ['\n', 'S', 'K', 'I', 'c']
	 next token: tensor([24]) ['L'] 
iter 5, the idx: tensor([[ 0, 31, 23, 21, 41, 24]]), decoded: ['\n', 'S', 'K', 'I', 'c', 'L']
	 next token: tensor([32]) ['T'] 
iter 6, the idx: tensor([[ 0, 31, 23, 21, 41, 24, 32]]), decoded: ['\n', 'S', 'K', 'I', 'c', 'L', 'T']
	 next token: tensor([11]) [';'] 
iter 7, the idx: tensor([[ 0, 31, 23, 21, 41, 24, 32, 11]]), decoded: ['\n', 'S', 'K', 'I', 'c', 'L', 'T', ';']


In [17]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [18]:
batch_size = 32
for steps in range(10000):
    # sample a batch of data
    xb, yb = get_batch('train')

    # step 1 - Forward pass
        # this automatically triggers m.forward() 
    logits, loss = m(xb, yb)

    # step 2 - reset gradients
        # if you forget the clear the gradients, the model will add the new gradients to the old ones
        # leading to massive gradient values and unstable training 
        # set_to_none sets the graident tensors to be None instead of filling them with 0's --> saves GPU memory + speeds up
    optimizer.zero_grad(set_to_none=True)

    # step 3 - backward pass (backpropagation)
        # traces backwards through every mathematical operations and calculates the gradient(derivative) for every param in m.parameters()
    loss.backward()

    # step 4 - update weights 
    optimizer.step()

print("loss is", loss.item())

loss is 2.4849228858947754


In [19]:
# rerunning again after training the model
starter_idx = torch.zeros((1, 1), dtype=torch.long)
print("starter_idx", starter_idx)
generated_chars = m.generate(idx=starter_idx, max_new_tokens=300)
decoded_chars = decode(generated_chars[0].tolist())
print("decoded_chars", "".join(decoded_chars)) # jibberish because the model is untrained 

starter_idx tensor([[0]])
iter 0, the idx: tensor([[0]]), decoded: ['\n']
	 next token: tensor([0]) ['\n'] 
iter 1, the idx: tensor([[0, 0]]), decoded: ['\n', '\n']
	 next token: tensor([35]) ['W'] 
iter 2, the idx: tensor([[ 0,  0, 35]]), decoded: ['\n', '\n', 'W']
	 next token: tensor([46]) ['h'] 
iter 3, the idx: tensor([[ 0,  0, 35, 46]]), decoded: ['\n', '\n', 'W', 'h']
	 next token: tensor([58]) ['t'] 
iter 4, the idx: tensor([[ 0,  0, 35, 46, 58]]), decoded: ['\n', '\n', 'W', 'h', 't']
	 next token: tensor([1]) [' '] 
iter 5, the idx: tensor([[ 0,  0, 35, 46, 58,  1]]), decoded: ['\n', '\n', 'W', 'h', 't', ' ']
	 next token: tensor([45]) ['g'] 
iter 6, the idx: tensor([[ 0,  0, 35, 46, 58,  1, 45]]), decoded: ['\n', '\n', 'W', 'h', 't', ' ', 'g']
	 next token: tensor([47]) ['i'] 
iter 7, the idx: tensor([[ 0,  0, 35, 46, 58,  1, 45, 47]]), decoded: ['\n', '\n', 'W', 'h', 't', ' ', 'g', 'i']
	 next token: tensor([39]) ['a'] 
iter 8, the idx: tensor([[ 0,  0, 35, 46, 58,  1, 45, 4

# The Mathematical trick in Self-Attention
`B, T, C = 4, 8, 2 # batch, time step, channels` 
- up to 8 tokens in a batch, and they're currently not taking to each other
- attention trick: 
    - token in the 5th location shouldn't commnicate with tokens in the 6th, 7th location etc 
    - token in the 5th location should only talk to the ones in the 4th, 3rd, 2nd, 1 st location
- aka information should only flow from previous context to current context 
- Shouldn't be getting any information from the future because we're trying to predict the future
- simpliest way would be to do an average of all the preceding elements 
- doing a sum or avg is a very weak form of interaction because the communication is very lossy
    - lost spacial arrangements of tokens etc


In [20]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2 # batch, time step, channels
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

The following serves to demo how torch.mean() works. torch.mean() takes in a tensor, and if you set dim=0(by setting 2nd param to be 0), it performs an average on all the 1st values of a time series(T). In our case, that would be avg(2, 4, 6) = 4. 

Similarily for 2nd value of the time series(T). So it becomes avg(4, 6, 8) = 6. 
So torch.mean(learn_mean_tensor.float(), 0) becomes tensor([4, 6])

In [21]:
learn_mean_tensor = torch.tensor([ [2,4], [4, 6], [6, 8]])
learn_mean_calculated = torch.mean(learn_mean_tensor.float(), 0)
print("learn_mean_tensor", learn_mean_tensor)
print("learn_mean_calculated", learn_mean_calculated)

learn_mean_tensor tensor([[2, 4],
        [4, 6],
        [6, 8]])
learn_mean_calculated tensor([4., 6.])


In this case, we are performing an average of all the token in the channel up till token nth. This is how attention is calculated. It's calculated such that the attention of the n-th token is the average of all the tokens leading up to and including the n-th token, in the 0th dim(aka, the column down).

Case token 0: token representation = [0.1808, -0.0700] 
xbow = [avg(0.1808), avg(-0.0700)]

Case token 1: token representation = 
{[ 0.1808, -0.0700],
[-0.3596, -0.9152]}
xbow = [avg(0.1808 + -0.3596), avg(-0.0700 + -0.9152)]
xbow = [-0.0894, -0.4926]


Case token 2: token representation = 
{[ 0.1808, -0.0700],
[-0.3596, -0.9152],
[ 0.6258,  0.0255]}
xbow = [avg(0.1808 + -0.3596 + 0.6258), avg(-0.0700 + -0.9152 + 0.0255)]
xbow = [0.1490, -0.3199]

...etc...
but for loops are computationally very inefficient ... can use leverage matrix multiplication to achieve the same result?

##### Bag of words casual attention computed using for loop!

In [22]:
# version - x-bag of words(xbow)
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # select that batch from 3D(B, T, C) and selects all tokens up till t-th token
        xbow[b, t] = torch.mean(xprev, 0)
print("Starting randint x", x[0])
print("Calculated xbow   ", xbow[0])

Starting randint x tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])
Calculated xbow    tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


Demo of the tril function. It is effectively setting all values above the specified diagonal in the matrix to be 0. If diagonal=0, then it will assume main diagonal. 

In [23]:
a = torch.ones(3,3)
b = torch.tril(input=a, diagonal=0)
print("a", a)
print("b", b)

a tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
b tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])


What if we encode the adding up till n-th token mechanism into matrix multiplication? We can leverage the tril function such that matrix multiplication between a=torch.tril(torch.ones(3,3), diagonal=0) and b=desired is effectively computing the same attention as above. The following is an example:
Do matrix mutiplication on your own to verify!

In [24]:
a = torch.tril(torch.ones(3,3), diagonal=0) 
b = torch.tensor([ [2,4], [4, 6], [6, 8] ]).float()
c = a @ b # this computes the sum
print("a= \n", a, "----\n", "b= \n", b, "----\n", "c= \n", c, "-----\n")

a= 
 tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]]) ----
 b= 
 tensor([[2., 4.],
        [4., 6.],
        [6., 8.]]) ----
 c= 
 tensor([[ 2.,  4.],
        [ 6., 10.],
        [12., 18.]]) -----



In [25]:
# tutorial demo
torch.manual_seed(42)
a = torch.tril(torch.ones(3,3))
a = a / torch.sum(a, 1, keepdim=True) # normalize by the sum of rows 
b = torch.randint(0, 10, (3,2)).float()
c = a @ b # now that we normalized a, c is the computed average instead of the sum
print('a=')
print(a)
print('-- \n b=')
print(b)
print('-- \n c=')
print(c)


a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
-- 
 b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
-- 
 c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


##### bag of words casual self-attention computed using tril

In [26]:
# version 2 - back to our bag of words
wei = torch.tril(torch.ones(T, T)) # wei stands for weights 
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) --> (B, T, C)
torch.allclose(xbow, xbow2, atol=1e-7, rtol=1e-5)

True

##### bag of words casual self-attention computed using softmax
- The weights begin as 0, and you can think of it as an interaction string, or as an affinity
- It's telling us how much of each token from the past do we want to aggregate(average up)
- masking all the token that has a value of 0 is saying that "tokens from the past can't communicate"
- "We will not aggregate anything from the tokens in the past" 
- xbow3 is the aggregation through matrix multiplication 
- this is the basis for the self-attention block

In [27]:
# version 3 - another way to produce bag of words using softmax --> end up using this in self-attention
# T by T(Time series aka # tokens in a batch) because it's relationship between the current token that 
# we're looking at and all the past tokens. There are a total of T tokens in a batch
tril = torch.tril(torch.ones(T, T))
# currently set to be 0 by us, but in the future, they will start looking at each other
# some tokens will find other tokens more or less interesting. Or interesting to different amounts(called affinity) 
wei = torch.zeros((T, T)) 
wei = wei.masked_fill(tril == 0, float('-inf')) # mask all values of 0 as -inf for softmax
# e^-inf = 0, so adding the masked -inf is effectively adding nothing
# so F.softmax's fraction formula is effectively giving wei the same matrix as above 
wei = F.softmax(wei, dim=-1) 
xbow3 = wei @ x
print("xbow vs xbow3", torch.allclose(xbow, xbow3, atol=1e-7, rtol=1e-5))
print("xbow2 vs xbow3", torch.allclose(xbow2, xbow3))

xbow vs xbow3 True
xbow2 vs xbow3 True


In [28]:
print("wei.shape", wei.shape)
wei

wei.shape torch.Size([8, 8])


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

##### Self-attention!
Every single token will emit 2 vectors
1. Query - What am I looking for?
2. Key - What do I contain? 
The way I get the affinity between these 2 tokens in a sequence is by performing a dot product between the key and query 
- So my query will dot product with all the keys
- that dot product now becomes a weight 
- if the query and key are very aligned(closed), the will interact in a high amount, and I'll get to learn more about that specific token, as opposed to other tokens in the sequence. 

We'll implement a single headed attention.
- A hyperparam involved is the head size 
- initializing linear modules, using bias = false, so these will apply matrix multiply with fixed weights 
- key --> will be (B, T, 16) because that's the head size 
- query(x) --> will be (B, T, 16)
- When I forward this linear on top of my x, all the tokens in all the positions in the B x T arrangements, are all parallel 
and independently produce a key and a query. So far, no communication happened yet 
- However, after all the queries has been made through nn.Linear, the communication comes now! 
- All the querries will dot product with all the keys 
- So now, wei will be defined as the affinity between q @ k, which will be (B, T, 16) @ (B, 16, T) --giving us---> (B, T, T) 

In [29]:
# version 4: self-attention
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
print("shape x.shape: ", x.shape)
print("shape x[0].shape: ", x[0].shape)
print("shape: x[0][0].shape", x[0][0].shape)

print("\n x", x[0])

shape x.shape:  torch.Size([4, 8, 32])
shape x[0].shape:  torch.Size([8, 32])
shape: x[0][0].shape torch.Size([32])

 x tensor([[ 1.8077e-01, -6.9988e-02, -3.5962e-01, -9.1520e-01,  6.2577e-01,
          2.5510e-02,  9.5451e-01,  6.4349e-02,  3.6115e-01,  1.1679e+00,
         -1.3499e+00, -5.1018e-01,  2.3596e-01, -2.3978e-01, -9.2111e-01,
          1.5433e+00,  1.3488e+00, -1.3964e-01,  2.8580e-01,  9.6512e-01,
         -2.0371e+00,  4.9314e-01,  1.4870e+00,  5.9103e-01,  1.2603e-01,
         -1.5627e+00, -1.1601e+00, -3.3484e-01,  4.4777e-01, -8.0164e-01,
          1.5236e+00,  2.5086e+00],
        [-6.6310e-01, -2.5128e-01,  1.0101e+00,  1.2155e-01,  1.5840e-01,
          1.1340e+00, -1.1539e+00, -2.9840e-01, -5.0754e-01, -9.2392e-01,
          5.4671e-01, -1.4948e+00, -1.2057e+00,  5.7182e-01, -5.9735e-01,
         -6.9368e-01,  1.6455e+00, -8.0299e-01,  1.3514e+00, -2.7592e-01,
         -1.5108e+00,  2.1048e+00,  2.7630e+00, -1.7465e+00,  1.4516e+00,
         -1.5103e+00,  8.2115e

### Attention Head Explained
#### nn.Linear(in_features, out_features) explained
- defines a linear transformation to the in_features such that it spits out out_features. In the following case, it takes in features size of C=32 to out_features size of head_size=16. In effect, it's performing a linear transformation of 32 -> 16
- it applies the following linear transformation y = xA_T + b, where x is the input and y is the output
- defining key = nn.Linear(C, head_size, bias=False) and then calling k = key(x) runs the forward pass of nn.Linear on x 
- In our context, this define a Linear Transf. of 32 feature channels to 16. It shrinks the **embedding dim**, which is the features per token, not the number of tokens! 
  - (B=4, T=8, C=32) defines 4 sequences(or batches), 8 tokens per batch/sequence, 32 features/token 
  - (B=4, T=8, C=16) defines something similar, but only 16 features 
- by projecting C=32 down to h=16, it reduces the memory and compute required per head, allowing the model to split C=32 into multiple parallel threads 
- bias=False just ignores the +b part 
- all weights are initialized as  uniform random numbers to start with

So x=(B=4, T=8, C=32)
- key(x) = (B=4, T=8, C=16) due to the effects of the linear transformation
- same for query(x)

#### Wei - affinity
wei = q @ k 
- Before - wei was constant, so it was applied the same way to all the batched elements 
- Now, every single batched element will have different weights because every single batched element contains different tokens

Before - without attention wei looks like this following softmax:
- notice how the softmax'ed probability is uniform distributed
```
wei.shape torch.Size([8, 8])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
```
After - with single head attention calculation, wei looks like this following softmax:
- notice how the the softmax'ed probability is no longer uniformly distributed
- this is because wei now takes into account the relationship between tokens using k @ q 
```
wei.shape torch.Size([4, 8, 8])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)
```
- For example - the 8th token knows what content it has, and what position it is in
- Now, the 8th token creates a query based on that, be like "hey, I'm looking for this kind of stuff"
- Query sounds like: 
  - That 8th token be like "I'm a vowel, I'm in the 8th position, and I'm looking for any consonants at positions up to 4" 
- Keys sound like:
  - "I'm a consonant, and I'm in a position up to 4", and that key will have a high number in that specific channel 
  - All the nodes gets to emit keys 
- So now, when the query and key dot product, they can find each other and create a high affinity 
- When they have a high affinity, through softmax, I will end up aggregating a lot of its info into my positon -> I get to learn a lot about it 

#### What x represents
- you can think of x as private info of this particular token 
- Ex: if I'm 5th token, and my info is kept in vector x, for the purpose of this single headed attention, I'm interested in
  - "What I have"
  - "If you find me interesting, here's what I'll communicate with you" --> stored in v 
  - V is the thing that gets aggregated for the purposes of this single head 
  - this is effectively the self-attention mechanism!

#### Attention
- Attention is a communicate mechanism --> allows the tokens to communicate with each other 
- A set of vectors in space that communicates. And if you want them to have a notion of space, you need to specifically add it. (Ex: Adding positional encoding to attention)
- Each example across batch dimension is processed completely independently and never "talk" to each other. "Batched matrix multiply" applies matrix multiplication in parallel across the batched dim. In our ex, the batch size is 4, we should really be saying that there are 4 seperate pools of 8 nodes, where those 8 nodes only talk to each other. i.e. 32 nodes that are being processed, so effectively 4 seperate pools of 8 

What if we want all the tokens to talk to each other?
Code that does upper triangular masking `wei = wei.masked_fill(tril == 0, float('-inf'))` 
- In an "encoder" attention block, just delete the single line that does the masking with tril, which allows all tokens to communicate. 
- What we have may be called "Decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like modeling. Will always have that line that does upper triangular masking 
- Attention allows both because it doesn't care. Attention supports arbitary connectivity between the node  

##### Self-attention vs Cross-attention
- Self-attention: because the Keys, query, and values are all attending to the same source, x, hence the nodes are self-attending. Aka, the nodes look at each other.
- Cross-attention: when there are seperate source of nodes to pull info from into our nodes. Aka, the nodes may look at other things besides each other 

##### Behind the scenes
Concrete example: A vowel looking for a consonant. Imagine training on a task where target token is a vowel at position i consistently relies on a preceding token, which is a consonant at position j to predict the next work. 
Summary: 
- vowel at position i Query(q_i) --replies on --> consonant at position j Key(k_j)
1. Forward pass(initial state), W_Q and W_K are random. The dot product of q_i * k_j is small or arbitrary. This means position i pays very little attention to position j. Effectively q @ k is small, so "attention" is small, so q pays very little "attention" to key
2. Backward Pass signal. The optimizer signals that A_{i, j} aka "attention" needs to be much higher
3. Updating Wq: the gradient adusts W_q so that whenever x_i, is a vowel, q_i projects into a region of vector space we can label as "seeking consonants" region
4. Updating Wk: The gradient adjusts W_k so that whever x_j is a consonant, k_j projects into the exact same region of vector space, acting as "I am a consonant" tag 
Over millions of setps across all token pairs in the dataset

### 3. Why Don't $W_Q$ and $W_K$ Collapse into the Same Matrix?
Because the dot product $q_i \cdot k_j^T$ is asymmetric with respect to sequence positions $i$ and $j$:Token $i$ is evaluating $q_i \cdot k_j^T$ to determine how much $i$ attends to $j$.Token $j$ is evaluating $q_j \cdot k_i^T$ to determine how much $j$ attends to $i$.Because English (and language in general) is directional and asymmetric—a verb looks for its object, but the object doesn't look for the verb in the same way—the required query vector for a token is almost never identical to its required key vector.

In [30]:
# let's see a single Head perform self-attention 
head_size = 16 
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x) # (B, T, 16)
print("k.shape: ", k.shape)
q = query(x) # (B, T, 16)
print("q.shape: ", q.shape)
# now, we want the communication between key and query 
wei = q @ k.transpose(-2, -1) # effectively performs (B, T, 16) @ (B, 16, T) --> (B, T, T), where B=4, T=8

print("weight.shape: ", wei.shape)
print("wei[0]", wei[0]) # raw output of the dot product, takes values -2 to 2, raw interaction, raw affinity between key and query

k.shape:  torch.Size([4, 8, 16])
q.shape:  torch.Size([4, 8, 16])
weight.shape:  torch.Size([4, 8, 8])
wei[0] tensor([[-1.7629, -1.3011,  0.5652,  2.1616, -1.0674,  1.9632,  1.0765, -0.4530],
        [-3.3334, -1.6556,  0.1040,  3.3782, -2.1825,  1.0415, -0.0557,  0.2927],
        [-1.0226, -1.2606,  0.0762, -0.3813, -0.9843, -1.4303,  0.0749, -0.9547],
        [ 0.7836, -0.8014, -0.3368, -0.8496, -0.5602, -1.1701, -1.2927, -1.0260],
        [-1.2566,  0.0187, -0.7880, -1.3204,  2.0363,  0.8638,  0.3719,  0.9258],
        [-0.3126,  2.4152, -0.1106, -0.9931,  3.3449, -2.5229,  1.4187,  1.2196],
        [ 1.0876,  1.9652, -0.2621, -0.3158,  0.6091,  1.2616, -0.5484,  0.8048],
        [-1.8044, -0.4126, -0.8306,  0.5898, -0.7987, -0.5856,  0.6433,  0.6303]],
       grad_fn=<SelectBackward0>)


In [31]:

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T, T)) they're no longer zeros coming from this key and query 
# masked the upper right half to enforce causality 
# --> aka, tokens can only look at current or past tokens, not ahead.
# Ex: look at row 3(representing token #2): it only looks at tokens 0, 1, and 2. It can't look at token 2+ 
# "They are not allowed to communicate"
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
# out = wei @ x # this being replaced by values

v = value(x)
out = wei @ v # v is the elements/vectors that we aggregate, instead of the raw x 

print("shape", out.shape)
print("wei[0] \n", wei[0])


shape torch.Size([4, 8, 16])
wei[0] 
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)



##### Scaled Attention 
- So if you have unit Gaussian inputs(mean=0, stdev=1), and you just perform wei naively, then you see that the variance of your wei will be on the order of head_size. In our case, it will be around 16. However, if you multiply by 1/sqrt(headsize), then the variance of wei will be around 1. 
- This applies even after you repeat this operation many times 
- This is important because wei feeds into softmax. It's very important that wei be fairly diffused. Why?
  - Due to softmax, if wei takes on very positive or very negative values, then softmax will converge towards 1 hot vectors. Demo below

In [32]:
# let's create a hypothetical unit gaussian simulation of what k, q, and wei could be
k_gaussian = torch.randn(B, T, head_size)
q_gaussian = torch.randn(B, T, head_size)
wei_gaussian = q_gaussian @ k_gaussian.transpose(-2, -1)
wei_gaussian_normaized = q_gaussian @ k_gaussian.transpose(-2, -1) * head_size**-0.5 # head_size**-0.5 is the square root
print(f'k.var: {k_gaussian.var()} \n')
print(f'q.var: {q_gaussian.var()} \n')
print(f'wei.var: {wei_gaussian.var()} \n')
print(f'wei_gaussian_normaized.var: {wei_gaussian_normaized.var()} \n')

k.var: 1.044861912727356 

q.var: 1.0700464248657227 

wei.var: 17.46897315979004 

wei_gaussian_normaized.var: 1.0918108224868774 



In [ ]:
# Let's say you're doing softmax of "evenly spaced out numbers
ex_weights = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])
print("softmax of evenly spaced out numbers", torch.softmax(ex_weights, dim=-1))

# So now, we take the exact same weights/distribution, and multiplies it 8x, 
# you'll see that softmax will begin to sharpen towards the max(in this case, it sharpened toward 0.2872)
# This is why we needed normalization --> to avoid the extremes otherwise softmax will be too "peaky" and you're basically 
# aggregating information into a single node  
print("softmax of not evenly spaced out numbers", torch.softmax(ex_weights*8, dim=-1))

softmax of evenly spaced out numbers tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])
softmax of not evenly spaced out numbers tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])


With 1 single head of attention, the current output looks like this: 
```
Whent whitridcowinen is by bth

Hiset bobe toe.
S:
O:
I thealilanss:
Want he uw hat vet?
F dilas ate
```
Looks slightly better than before, which was almost just randomness. However, we can do a better job using multihead attention! 